In [ ]:
Task 4 : Statistical Tests:-

Now let’s move on to statistical testing using the Adult Census Income dataset. The purpose of this task is to statistically validate the
hypotheses created in Task 3. While charts and graphs show patterns, statistical tests help confirm whether these patterns are real or 
happened by chance.
 Hypothesis Testing:-
   When we test a hypothesis, we start with 2 statements:
    1. H₀ (Null hypothesis): “Nothing interesting is happening.”
       No relationship, no difference, no association.
    2. H₁ (Alternative hypothesis): “Something is happening.”
       There is a relationship, difference, or association.
    Example (Education vs Income):
    H₀: Education and income are independent (education doesn’t affect income category).
    H₁: Education and income are associated.
        
 p-value:-
    This value helps us decide whether our results are meaningful. A small p-value means the results are unlikely to happen by
    random chance if the null hypothesis were true, which gives us reason to question or reject it.
The common rule
    If p ≤ 0.05, we say it is statistically significant and reject H₀.     
    If p > 0.05, we say it is not statistically significant and fail to reject H₀. 
        
Type I and Type II Errors:-
   Statistical testing can lead to two types of errors:
      Type I Error (False Positive):
        Rejecting the null hypothesis when it is actually true.
        Example: Concluding education affects income when it really does not.
      Type II Error (False Negative):
        Failing to reject the null hypothesis when it is actually false.
        Example: Missing a real relationship between education and income.
   These errors highlight why statistical conclusions must be interpreted carefully and within context.
      trick:  Type I = “false alarm” , Type II = “missed detection” 

After covering these basics, the next step is to choose the correct statistical test for each hypothesis and analyze the data. 
    A) Categorical feature vs Categorical target (income)
       Use Chi-square test of independence
       Used when both variables are categorical.
         Education vs Income
         Workclass vs Income
         Occupation vs Income
         Marital Status vs Income
         Relationship vs Income
         Sex vs Income
           
    B) Numeric feature vs Binary target (income)
       Compare the numeric feature between income groups using:
       Welch’s t-test (Used when comparing the mean of a numerical variable between two groups)
           Age vs Income
           Education Number vs Income
           Hours per Week vs Income
           Capital Gain vs Income
           Capital Loss vs Income
           fnlwgt vs Income
       Welch’s t-test is preferred because it does not assume equal variance between groups.
           
       Mann–Whitney U (Used as a robustness check for skewed numerical variables.)
           Capital Gain
           Capital Loss
       This non-parametric test compares distributions rather than means.               

In simple terms, statistical tests help us decide whether what we see in the data is meaningful or just random noise.

In [1]:
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu

In [5]:
# Column names (same order as the UCI Adult dataset)
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]
def load_adult(path: str, is_test: bool = False) -> pd.DataFrame:
    """Load Adult data from UCI-style files (adult.data, adult.test).
    What this handles:
    - No header row (so we add column names ourselves)
    - Trims extra spaces in text columns
    - Converts '?' into real missing values (NaN)
    - Removes '.' from income labels in adult.test (e.g., '>50K.' → '>50K')
    - Keeps only valid target labels
    """
    if is_test:
        # adult.test has a header-like first row, so we skip it
        df = pd.read_csv(path, names=COLUMNS, sep=",", skipinitialspace=True, skiprows=1)
    else:
        df = pd.read_csv(path, names=COLUMNS, sep=",", skipinitialspace=True)

    # Clean text columns: strip spaces so categories don’t split incorrectly
    for c in df.select_dtypes(include=["object"]).columns:
        df[c] = df[c].astype(str).str.strip()

    # Convert '?' into proper missing values
    df.replace("?", np.nan, inplace=True)

    # Fix income labels in test data (some have trailing '.')
    df["income"] = df["income"].str.replace(".", "", regex=False).str.strip()

    # Keep only valid labels
    df = df[df["income"].isin(["<=50K", ">50K"])].copy()
    return df

TRAIN_PATH = r"C:\suchi\ML-assignment\adult\adult.data"
TEST_PATH  = r"C:\suchi\ML-assignment\adult\adult.test"

train_df = load_adult(TRAIN_PATH, is_test=False)
test_df  = load_adult(TEST_PATH, is_test=True)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain income counts:\n", train_df["income"].value_counts())
print("\nTest income counts:\n", test_df["income"].value_counts())

Train shape: (32561, 15)
Test shape : (16281, 15)

Train income counts:
 income
<=50K    24720
>50K      7841
Name: count, dtype: int64

Test income counts:
 income
<=50K    12435
>50K      3846
Name: count, dtype: int64


In [6]:
#Helper Functions
def chi_square_test(df: pd.DataFrame, feature: str) -> dict:
    """Chi-square test of independence between a categorical feature and income."""
    tmp = df[[feature, "income"]].dropna()
    table = pd.crosstab(tmp[feature], tmp["income"])
    chi2, p, dof, expected = chi2_contingency(table)
    return {"feature": feature, "test": "Chi-square", "chi2": chi2, "dof": dof, "p_value": p, "table": table}

def welch_t_test(df: pd.DataFrame, feature: str) -> dict:
    """Welch t-test comparing numeric feature means between income groups."""
    tmp = df[[feature, "income"]].dropna()
    x = tmp.loc[tmp["income"] == "<=50K", feature].astype(float).to_numpy()
    y = tmp.loc[tmp["income"] == ">50K",  feature].astype(float).to_numpy()
    t_stat, p = ttest_ind(x, y, equal_var=False)  # Welch
    return {
        "feature": feature,
        "test": "Welch t-test",
        "t_stat": t_stat,
        "p_value": p,
        "mean_<=50K": float(np.mean(x)),
        "mean_>50K": float(np.mean(y)),
        "n_<=50K": len(x),
        "n_>50K": len(y),
    }

def mann_whitney_test(df: pd.DataFrame, feature: str) -> dict:
    """Mann–Whitney U test (non-parametric) for numeric feature difference."""
    tmp = df[[feature, "income"]].dropna()
    x = tmp.loc[tmp["income"] == "<=50K", feature].astype(float).to_numpy()
    y = tmp.loc[tmp["income"] == ">50K",  feature].astype(float).to_numpy()
    u_stat, p = mannwhitneyu(x, y, alternative="two-sided")
    return {
        "feature": feature,
        "test": "Mann–Whitney U",
        "u_stat": u_stat,
        "p_value": p,
        "median_<=50K": float(np.median(x)),
        "median_>50K": float(np.median(y)),
    }

In [ ]:
1. Apply Statistical Tests to our Generated Hypotheses (Train Data)
    We’ll test the major Task 3 hypothesis areas.

In [7]:
# 1.1 Education vs Income (Chi-square)
edu_res = chi_square_test(train_df, "education")
print("Education vs Income")
print("Test:", edu_res["test"])
print("p-value:", edu_res["p_value"])
print("\nContingency table preview:")
edu_res["table"].head(10)

Education vs Income
Test: Chi-square
p-value: 0.0

Contingency table preview:


income,<=50K,>50K
education,,
10th,871,62
11th,1115,60
12th,400,33
1st-4th,162,6
5th-6th,317,16
7th-8th,606,40
9th,487,27
Assoc-acdm,802,265
Assoc-voc,1021,361


In [ ]:
Education
Education shows a statistically significant association with income. Higher education levels are strongly linked to earning more than $50,000.
The null hypothesis was rejected.

In [8]:
#1.2 Age vs Income (Welch t-test)
age_res = welch_t_test(train_df, "age")
age_res

{'feature': 'age',
 'test': 'Welch t-test',
 't_stat': np.float64(-50.264210024707836),
 'p_value': np.float64(0.0),
 'mean_<=50K': 36.78373786407767,
 'mean_>50K': 44.24984058155847,
 'n_<=50K': 24720,
 'n_>50K': 7841}

In [ ]:
Age
The mean age differs significantly between income groups. Higher-income individuals tend to be older, supporting the hypothesis
that experience influences earnings.

In [9]:
#1.3 Hours per Week vs Income (Welch t-test)
hours_res = welch_t_test(train_df, "hours_per_week")
hours_res

{'feature': 'hours_per_week',
 'test': 'Welch t-test',
 't_stat': np.float64(-45.123095093109875),
 'p_value': np.float64(0.0),
 'mean_<=50K': 38.840210355987054,
 'mean_>50K': 45.473026399693914,
 'n_<=50K': 24720,
 'n_>50K': 7841}

In [ ]:
Hours per Week
Individuals working longer hours are significantly more likely to earn above $50,000. The difference in average working hours between
income groups is statistically significant.

In [10]:
#1.4 Workclass + Occupation vs Income (Chi-square)
workclass_res = chi_square_test(train_df, "workclass")
occupation_res = chi_square_test(train_df, "occupation")

print("Workclass p-value:", workclass_res["p_value"])
print("Occupation p-value:", occupation_res["p_value"])

Workclass p-value: 1.9338476684848218e-174
Occupation p-value: 0.0


In [ ]:
Occupation and Workclass
Both occupation and workclass show strong associations with income. Professional and managerial roles, as well as certain workclass
categories, are more likely to fall into the higher income group.

In [11]:
#1.5 Marital Status + Relationship vs Income (Chi-square)
marital_res = chi_square_test(train_df, "marital_status")
relationship_res = chi_square_test(train_df, "relationship")

print("Marital status p-value:", marital_res["p_value"])
print("Relationship p-value:", relationship_res["p_value"])

Marital status p-value: 0.0
Relationship p-value: 0.0


In [ ]:
Marital Status and Relationship
Marital status and relationship type are statistically associated with income. Married individuals, especially those classified as
“Married-civ-spouse,” show higher proportions of earners above $50,000.

In [12]:
#1.6 Capital Gain/Loss vs Income (Welch t-test + Mann–Whitney U)
cap_gain_t = welch_t_test(train_df, "capital_gain")
cap_gain_u = mann_whitney_test(train_df, "capital_gain")

cap_loss_t = welch_t_test(train_df, "capital_loss")
cap_loss_u = mann_whitney_test(train_df, "capital_loss")

print("Capital Gain (Welch t-test) p-value:", cap_gain_t["p_value"])
print("Capital Gain (Mann–Whitney) p-value:", cap_gain_u["p_value"])

print("\nCapital Loss (Welch t-test) p-value:", cap_loss_t["p_value"])
print("Capital Loss (Mann–Whitney) p-value:", cap_loss_u["p_value"])

Capital Gain (Welch t-test) p-value: 2.242936993869241e-117
Capital Gain (Mann–Whitney) p-value: 0.0

Capital Loss (Welch t-test) p-value: 3.786929420630867e-89
Capital Loss (Mann–Whitney) p-value: 7.021293579330328e-143


In [ ]:
Capital Gain and Capital Loss
Capital gain is one of the strongest indicators of high income. Individuals reporting any capital gain are far more likely to earn
above $50,000. Results were statistically significant across both parametric and non-parametric tests.

In [13]:
#1.7 Sex + Race vs Income (Chi-square) 
sex_res = chi_square_test(train_df, "sex")
race_res = chi_square_test(train_df, "race")

print("Sex p-value:", sex_res["p_value"])
print("Race p-value:", race_res["p_value"])

Sex p-value: 0.0
Race p-value: 2.305960610160958e-70


In [ ]:
Demographic Variables (Sex and Race)
Statistical tests show differences in income distribution across sex and race categories. These findings must be interpreted cautiously,
as they reflect broader societal and historical factors rather than individual ability.

In [14]:
#1.8 fnlwgt vs Income (Welch t-test)
fnlwgt_res = welch_t_test(train_df, "fnlwgt")
fnlwgt_res

{'feature': 'fnlwgt',
 'test': 'Welch t-test',
 't_stat': np.float64(1.7412056101999103),
 'p_value': np.float64(0.08167013125422205),
 'mean_<=50K': 190340.8651699029,
 'mean_>50K': 188005.0,
 'n_<=50K': 24720,
 'n_>50K': 7841}

In [ ]:
fnlwgt (Census Weight)
The census weight variable does not show a strong or meaningful relationship with income. Its primary purpose is survey weighting,
not income prediction.

In [ ]:
Put All Results Into One Table
  This table helps you quickly see:
    which tests are significant (p ≤ 0.05)
    direction of difference for numeric features (means by income group)

In [15]:
results = []

# Categorical features (Chi-square)
for f in ["education", "workclass", "occupation", "marital_status", "relationship", "sex", "race"]:
    r = chi_square_test(train_df, f)
    results.append({
        "feature": f,
        "feature_type": "categorical",
        "test": r["test"],
        "p_value": r["p_value"]
    })

# Numeric features (Welch t-test)
for f in ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss", "fnlwgt"]:
    r = welch_t_test(train_df, f)
    results.append({
        "feature": f,
        "feature_type": "numeric",
        "test": r["test"],
        "p_value": r["p_value"],
        "mean_<=50K": r["mean_<=50K"],
        "mean_>50K": r["mean_>50K"]
    })

summary_df = pd.DataFrame(results).sort_values("p_value")
summary_df

,feature,feature_type,test,p_value,mean_<=50K,mean_>50K
0,education,categorical,Chi-square,0.000000e+00,NaN,NaN
2,occupation,categorical,Chi-square,0.000000e+00,NaN,NaN
3,marital_status,categorical,Chi-square,0.000000e+00,NaN,NaN
4,relationship,categorical,Chi-square,0.000000e+00,NaN,NaN
5,sex,categorical,Chi-square,0.000000e+00,NaN,NaN
7,age,numeric,Welch t-test,0.000000e+00,36.783738,44.249841
8,education_num,numeric,Welch t-test,0.000000e+00,9.595065,11.611657
9,hours_per_week,numeric,Welch t-test,0.000000e+00,38.840210,45.473026
1,workclass,categorical,Chi-square,1.933848e-174,NaN,NaN
10,capital_gain,numeric,Welch t-test,2.242937e-117,148.752468,4006.142456


In [ ]:
In Task 4, we validated our Task 3 hypotheses using statistical tests:
  Chi-square for categorical features (education, occupation, workclass, etc.)
  Welch’s t-test for numeric features (age, hours/week, etc.)
  Mann–Whitney U as a robustness check for skewed numeric features (capital_gain/loss)
  we used p-values to decide whether to reject or fail to reject H₀, and summarized results in one table.

In [ ]:
Conclusion:-
The statistical tests support many of the patterns we first noticed during data exploration. Features such as education level, age, number of
working hours, occupation, marital status, and capital gain all show a clear relationship with income. Among these, education and capital gain
stand out as especially important factors in determining whether a person earns more than $50,000.
The analysis also shows that income is not evenly distributed across different demographic groups. These differences should be interpreted
carefully and fairly, especially when using the data to build prediction models. The fnlwgt variable, which represents census sampling weight,
does not show a strong connection with income.
Overall, these statistical results help us decide which features are useful and provide a solid starting point for building machine learning
models in the next step of the project.
